# Yahoo Finance Data Extraction & DVC Tracking

This notebook demonstrates how to fetch data using `yfinance` and then use our custom `src/data/dvc_storage.py` module to store the resulting DataFrames securely in the Oracle Cloud DVC remote.

**Prerequisite:** Make sure you have successfully authenticated with Oracle via DVC (run `00-colab-setup.ipynb` if you are in Colab).

In [ ]:
import yfinance as yf
import pandas as pd
import sys
import os

# Ensure the 'src' directory is accessible for imports
sys.path.append(os.path.abspath('..'))

from src.data.dvc_storage import store_dataframes_in_dvc

In [ ]:
# Example 1: Fetch and store a single ticker
ticker_name = 'MSFT'
print(f"📉 Fetching 1y data for {ticker_name}...")
msft_df = yf.Ticker(ticker_name).history(period='1y')

if not msft_df.empty:
    # We pass the single DataFrame and its name to our storage function
    store_dataframes_in_dvc(data=msft_df, names=ticker_name, folder_type='raw')
else:
    print(f"❌ No data returned for {ticker_name}")

In [ ]:
# Example 2: Fetch and store multiple tickers simultaneously as a list
tickers_list = ['GOOGL', 'AMZN']
dfs = []

print(f"\n📉 Fetching 1mo data for multiple tickers: {tickers_list}...")
for t in tickers_list:
    df = yf.Ticker(t).history(period='1mo')
    dfs.append(df)

# Ensure we have valid data before storing
valid_dfs = [df for df in dfs if not df.empty]
valid_names = [t for t, df in zip(tickers_list, dfs) if not df.empty]

if valid_dfs:
    # We pass the list of DataFrames and the list of names to our storage function
    store_dataframes_in_dvc(data=valid_dfs, names=valid_names, folder_type='raw')
else:
    print("❌ No valid data returned for the tickers list.")